In [ ]:
import pandas as pd

In [ ]:
data = pd.read_csv('data/adult.csv')

In [ ]:
data.head()

In [ ]:
data.isnull().sum()

In [ ]:
data['income'].unique()

In [ ]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
le = LabelEncoder()
ohe = OneHotEncoder()

In [ ]:
le.fit(data['income'])
y = le.transform(data['income'])

In [ ]:
features_columns = [c for c in data.columns if c != 'income']
data[features_columns].head(2)

In [ ]:
data = data[features_columns]

In [ ]:
selectedColumns = data[ ['age', 'education', 'educational-num',  'marital-status', 'gender', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country'] ]

In [ ]:
X = pd.get_dummies(selectedColumns)

**????? ?????????? ???????????????**

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 1)

In [ ]:
from sklearn.model_selection import GridSearchCV

params = [ {'max_depth': list( range(1, 20)),  'n_estimators': list( range(1, 20))} ]
gs = GridSearchCV( RandomForestClassifier(), param_grid = params, scoring = 'accuracy', return_train_score = False )

In [ ]:
gs.fit( X, y )
gs.best_params_

In [ ]:
params = [ {'max_depth': list( range(1, 20) )} ]
gs = GridSearchCV( DecisionTreeClassifier(), param_grid = params, scoring = 'accuracy', return_train_score = False )

In [ ]:
gs.fit( X, y )
gs.best_params_

In [ ]:
models = []
names = ['LogisticRegression','Random Forest','Neural Network','DecisionTreeClassifier','SVM',]

models.append((LogisticRegression(max_iter=1000,n_jobs=-1)))
models.append((RandomForestClassifier(n_estimators=14, max_depth=15,n_jobs=-1)))
models.append((MLPClassifier(activation='logistic', max_iter=1000, nesterovs_momentum=True)))
models.append((DecisionTreeClassifier(max_depth=9)))
models.append((SVC()))

In [ ]:
print (models)

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score

for i in range(0,len(models)):    
    cv_result = cross_val_score(models[i],X_train,y_train,cv=5,scoring='accuracy').mean()
    score=models[i].fit(X_train,y_train)
    prediction = models[i].predict(X_test)
    acc_score = accuracy_score(y_test,prediction)     
    print ('-'*40)
    print ('{0}: {1}'.format(names[i], acc_score))

### ???????? ???????? AUC ??? ????????? ??????? ? ?? ??????????

**Logistic Regression**

????????????? - ????????? ?????? (????????? ?????? ????????, ??? ????? ?????? ???????? ????????????)

In [ ]:
#C - ???????? ?????????????. ???? ????????????? ??????? ??????????????? C
model_1_lr = LogisticRegression(solver='liblinear' ).fit( X_train, y_train )
predictions_1_lr = model_1_lr.predict_proba(X_test)

In [ ]:
model_2_lr = LogisticRegression(penalty = 'l1', C = 0.01, solver='liblinear' ).fit( X_train, y_train )
predictions_2_lr = model_2_lr.predict_proba(X_test)

In [ ]:
model_3_lr = LogisticRegression(penalty = 'l2', C = 0.01, solver='liblinear' ).fit( X_train, y_train )
predictions_3_lr = model_3_lr.predict_proba(X_test)

In [ ]:
from sklearn.metrics import roc_curve

false_positive_rate_1_lr, true_positive_rate_1_lr, thresholds_1_lr = roc_curve( y_test, predictions_1_lr[:, 1] )
false_positive_rate_2_lr, true_positive_rate_2_lr, thresholds_2_lr = roc_curve( y_test, predictions_2_lr[:, 1] )
false_positive_rate_3_lr, true_positive_rate_3_lr, thresholds_3_lr = roc_curve( y_test, predictions_3_lr[:, 1] )

In [ ]:
from matplotlib import pyplot as plt

plt.plot( false_positive_rate_1_lr, true_positive_rate_1_lr, label = 'no penalty' )
plt.plot( false_positive_rate_2_lr, true_positive_rate_2_lr, label = 'l1' )
plt.plot( false_positive_rate_3_lr, true_positive_rate_3_lr, label = 'l2' )

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')

plt.legend( loc = 0 )
plt.show()

In [ ]:
from sklearn.metrics import roc_auc_score

a = roc_auc_score( y_test, predictions_1_lr[:, 1] ).astype(str)
b = roc_auc_score( y_test, predictions_2_lr[:, 1] ).astype(str)
c = roc_auc_score( y_test, predictions_3_lr[:, 1] ).astype(str)

print(a+' '+ b+ ' '+ c)

**Neural Network**

In [ ]:
#alpha=0.001 = L2 penalty (regularization term) parameter.
from numpy import mean

alpha = []
cross_val_scores_MLP=[]
predictions_MLP=[]

for i in range(0, 5): 
    MLP = MLPClassifier(alpha = i * 0.25, activation='logistic', max_iter=1000, nesterovs_momentum=True) 
    MLP.fit(X_train, y_train)
    predictions_MLP.append(MLP.predict_proba(X_test))
    scores = cross_val_score(MLP, X, y, cv = 5) 
    avg_cross_val_score = mean(scores)*100
    cross_val_scores_MLP.append(avg_cross_val_score) 
    alpha.append(i * 0.25) 

In [ ]:
for i in range(0, len(alpha)): 
    print(str(alpha[i])+' : '+str(cross_val_scores_MLP[i]))

In [ ]:
predictions_MLP[0][:,1]

In [ ]:
fpr = [None] * len(alpha)
tpr = [None] * len(alpha)
thres = [None] * len(alpha)


for i in range(len(alpha)):
    fpr[i], tpr[i], thres[i] = (roc_curve( y_test, predictions_MLP[i][:,1] ))

In [ ]:
plt.plot( fpr[0], tpr[0], label = '{1} {0}'.format('no regularization', alpha[0]) )
plt.plot( fpr[1], tpr[1], label = '{1} {0}'.format('l2', alpha[1]) )
plt.plot( fpr[2], tpr[2], label = '{1} {0}'.format('l2', alpha[2]) )
plt.plot( fpr[3], tpr[3], label = '{1} {0}'.format('l2', alpha[3]) )
plt.plot( fpr[4], tpr[4], label = '{1} {0}'.format('l2', alpha[4]) )

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')

plt.legend( loc = 0 )
plt.show()

**SVM**

In [ ]:
#penalty = L2 penalty (regularization term) parameter.
#C - ???????? ?????????????. ???? ????????????? ??????? ??????????????? C
model_1_svc = SVC(probability=True).fit( X_train, y_train )
predictions_1_svc = model_1_svc.predict_proba(X_test)

In [ ]:
model_2_svc = SVC(penalty = 'l1', C = 0.01, probability=True).fit( X_train, y_train )
predictions_2_svc = model_2_svc.predict_proba(X_test)

In [ ]:
model_3_svc = SVC(penalty = 'l2', C = 0.01, probability=True).fit( X_train, y_train )
predictions_3_svc = model_3_svc.predict_proba(X_test)

In [ ]:
from sklearn.metrics import roc_curve

false_positive_rate_1_svc, true_positive_rate_1_svc, thresholds_1_svc = roc_curve( y_test, predictions_1_svc[:, 1] )
false_positive_rate_2_svc, true_positive_rate_2_svc, thresholds_2_svc = roc_curve( y_test, predictions_2_svc[:, 1] )
false_positive_rate_3_svc, true_positive_rate_3_svc, thresholds_3_svc = roc_curve( y_test, predictions_3_svc[:, 1] )

In [ ]:
plt.plot( false_positive_rate_1_svc, true_positive_rate_1_svc, label = 'no penalty' )
plt.plot( false_positive_rate_2_svc, true_positive_rate_2_svc, label = 'l1' )
plt.plot( false_positive_rate_3_svc, true_positive_rate_3_svc, label = 'l2' )

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')

plt.legend( loc = 0 )
plt.show()

In [ ]:
a = roc_auc_score( y_test, predictions_1_svc[:, 1] ).astype(str)
b = roc_auc_score( y_test, predictions_2_svc[:, 1] ).astype(str)
c = roc_auc_score( y_test, predictions_3_svc[:, 1] ).astype(str)

print(a+' '+ b+ ' '+ c)

**Random Forest Classifier**

In [ ]:
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

In [ ]:
model_RFC = RandomForestClassifier(n_estimators=14, max_depth=15, n_jobs=-1).fit( X_train, y_train )
predictions_RFC = model_RFC.predict_proba(X_test)

In [ ]:
prediction = model_RFC.predict(X_test)

print ('-'*40)
print ('Accuracy score:')
print (accuracy_score(y_test,prediction))
print ('-'*40)
print ('Confusion Matrix:')
print (confusion_matrix(y_test,prediction))
print ('-'*40)
print ('Classification Matrix:')
print (classification_report(y_test,prediction))